# HAR-RV replication for USD/CHF

Construction of daily realized volatility from high-frequency USD/CHF data, HAR-RV estimation, inference, comparison with AR(22), and one-step-ahead in-sample forecasting.

Detailed mathematical explanations are provided in the project README.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np 
import statsmodels.api as sm
from scipy.stats import f

## 1. Load and prepare tick data

Load all USD/CHF tick files, construct the log-mid price, remove weekend observations, and define trading days.


In [2]:
download_dir = Path(r"C:\Users\maria\Project\HAR-RV\download")

# Read every downloaded USD/CHF tick-data CSV
files = sorted(download_dir.glob("*.csv"))

if not files:
    raise FileNotFoundError(f"No CSV files found in {download_dir}")

print("Files found:")
for file in files:
    print(" -", file.name)

# Merge all raw tick files into one DataFrame
tick_data = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

# Remove any duplicated boundary ticks and put everything in time order
tick_data = (
    tick_data
    .drop_duplicates(subset=["timestamp", "askPrice", "bidPrice"])
    .sort_values("timestamp")
    .reset_index(drop=True)
)

tick_data["timestamp"] = pd.to_datetime(tick_data["timestamp"], unit="ms", utc=True)
tick_data["logMidPrice"] = (np.log(tick_data["askPrice"]) + np.log(tick_data["bidPrice"])) / 2

print(f"Total ticks: {len(tick_data):,}")
print(f"From {tick_data['timestamp'].min()} to {tick_data['timestamp'].max()}")


Files found:
 - usdchf-tick-2022-01-01-2022-07-01.csv
 - usdchf-tick-2022-07-01-2023-01-01.csv
 - usdchf-tick-2023-01-01-2023-07-01.csv
 - usdchf-tick-2023-07-01-2024-01-01.csv
 - usdchf-tick-2024-01-01-2024-07-01.csv
 - usdchf-tick-2024-07-01-2025-01-01.csv
 - usdchf-tick-2025-01-01-2025-07-01.csv
 - usdchf-tick-2025-07-01-2026-01-01.csv
 - usdchf-tick-2026-01-01-2026-07-01.csv
Total ticks: 78,604,103
From 2022-01-02 22:10:02.761000+00:00 to 2026-06-30 23:59:58.223000+00:00


In [3]:
dt = tick_data["timestamp"]

weekend_mask = (
    ((dt.dt.day_name() == "Friday") & (dt.dt.hour >= 21)) |
    (dt.dt.day_name() == "Saturday") |
    ((dt.dt.day_name() == "Sunday") & (dt.dt.hour < 22))
)

tick_data = tick_data.loc[~weekend_mask].copy()

tick_data["trading_date"] = (
    tick_data["timestamp"] + pd.Timedelta(hours=2)
).dt.date

tick_data["logReturn"] = tick_data.groupby("trading_date")["logMidPrice"].diff()

In [4]:
tick_data.head()

,timestamp,askPrice,bidPrice,logMidPrice,trading_date,logReturn
0,2022-01-02 22:10:02.761000+00:00,0.91303,0.91113,-0.092028,2022-01-03,NaN
1,2022-01-02 22:10:18.608000+00:00,0.91294,0.91113,-0.092077,2022-01-03,-0.000049
2,2022-01-02 22:10:30.823000+00:00,0.91294,0.91117,-0.092055,2022-01-03,0.000022
3,2022-01-02 22:10:34.624000+00:00,0.91241,0.91150,-0.092165,2022-01-03,-0.000109
4,2022-01-02 22:10:34.725000+00:00,0.91238,0.91136,-0.092258,2022-01-03,-0.000093


## 2. Construct daily realized volatility

Compute fast and slow realized variance and combine them using the two-scale correction.


In [5]:
rv_daily = (
    tick_data
    .sort_values("timestamp")
    .assign(logReturn_daily=lambda x: x.groupby("trading_date")["logMidPrice"].diff())
    .dropna(subset=["logReturn_daily"])
    .groupby("trading_date", as_index=False)
    .agg(RV_fast=("logReturn_daily", lambda r: (r ** 2).sum()))
)

In [6]:
K = 10

slow_daily_records = []

for trading_date, day in tick_data.groupby("trading_date", sort=False):
    prices = (
        day.sort_values("timestamp")["logMidPrice"]
        .to_numpy(dtype=np.float64, copy=False)
    )

    rv_by_grid = []
    n_by_grid = []

    for offset in range(K):
        returns = np.diff(prices[offset::K])
        rv_by_grid.append(np.dot(returns, returns))
        n_by_grid.append(returns.size)

    slow_daily_records.append({
        "trading_date": trading_date,
        "RV_slow": np.mean(rv_by_grid),
        "n_bar": np.mean(n_by_grid),
    })

rv_slow_daily = pd.DataFrame(slow_daily_records)

rv_daily = (
    rv_daily
    .drop(columns=["RV_slow", "n_bar"], errors="ignore")
    .merge(rv_slow_daily, on="trading_date", how="left")
)

rv_daily

,trading_date,RV_fast,RV_slow,n_bar
0,2022-01-03,1.550085e-05,1.858745e-05,3628.9
1,2022-01-04,4.423198e-05,2.027893e-05,4673.8
2,2022-01-05,3.009169e-05,1.947776e-05,3715.4
3,2022-01-06,1.940771e-05,1.816665e-05,4636.0
4,2022-01-07,1.200306e-05,1.700804e-05,4088.0
...,...,...,...,...
1165,2026-06-25,1.406106e-05,1.867974e-05,6257.4
1166,2026-06-26,1.087954e-05,1.573885e-05,6130.6
1167,2026-06-29,7.859088e-06,9.494561e-06,4513.6
1168,2026-06-30,2.267291e-05,1.456705e-05,5335.2


In [7]:
n_daily = (
    tick_data
    .sort_values("timestamp")
    .assign(fast_return=lambda x: x.groupby("trading_date")["logMidPrice"].diff())
    .dropna(subset=["fast_return"])
    .groupby("trading_date", as_index=False)
    .size()
    .rename(columns={"size": "n"})
)

rv_daily = rv_daily.merge(n_daily, on="trading_date", how="left")

rv_daily["IV_TS"] = (
    rv_daily["RV_slow"] - (rv_daily["n_bar"] / rv_daily["n"]) * rv_daily["RV_fast"]
)
rv_daily = rv_daily.drop(columns=["n", "n_bar"])
rv_daily["RV"] = np.sqrt(rv_daily["IV_TS"])
rv_daily

,trading_date,RV_fast,RV_slow,IV_TS,RV
0,2022-01-03,1.550085e-05,1.858745e-05,1.703775e-05,0.004128
1,2022-01-04,4.423198e-05,2.027893e-05,1.585658e-05,0.003982
2,2022-01-05,3.009169e-05,1.947776e-05,1.646932e-05,0.004058
3,2022-01-06,1.940771e-05,1.816665e-05,1.622626e-05,0.004028
4,2022-01-07,1.200306e-05,1.700804e-05,1.580799e-05,0.003976
...,...,...,...,...,...
1165,2026-06-25,1.406106e-05,1.867974e-05,1.727384e-05,0.004156
1166,2026-06-26,1.087954e-05,1.573885e-05,1.465105e-05,0.003828
1167,2026-06-29,7.859088e-06,9.494561e-06,8.708809e-06,0.002951
1168,2026-06-30,2.267291e-05,1.456705e-05,1.230014e-05,0.003507


## 3. Estimate the HAR-RV model

Construct daily, weekly, and monthly volatility regressors and estimate the one-day-ahead HAR model by OLS.


In [8]:
rv_features = rv_daily.sort_values("trading_date").copy()

rv_features["RV_w"] = rv_features["RV"].rolling(window=5).mean()
rv_features["RV_m"] = rv_features["RV"].rolling(window=22).mean()

rv_features = (
    rv_features[["trading_date", "RV", "RV_w", "RV_m"]]
    .dropna(subset=["RV_w", "RV_m"])
    .reset_index(drop=True)
)

rv_features

,trading_date,RV,RV_w,RV_m
0,2022-02-01,0.003628,0.003955,0.003864
1,2022-02-02,0.003159,0.003800,0.003820
2,2022-02-03,0.004801,0.003851,0.003857
3,2022-02-04,0.005221,0.004146,0.003910
4,2022-02-07,0.003379,0.004038,0.003881
...,...,...,...,...
1144,2026-06-25,0.004156,0.003754,0.003785
1145,2026-06-26,0.003828,0.003867,0.003812
1146,2026-06-29,0.002951,0.003651,0.003773
1147,2026-06-30,0.003507,0.003653,0.003761


In [9]:
har_df = (
    rv_features
    .sort_values("trading_date")
    .assign(RV_next=lambda x: x["RV"].shift(-1))
    .rename(columns={"RV": "RV_d"})
    [["trading_date", "RV_next", "RV_d", "RV_w", "RV_m"]]
    .dropna()
    .reset_index(drop=True)
)

X = har_df[["RV_d", "RV_w", "RV_m"]]
X = sm.add_constant(X)  # adds intercept column of 1s

y = har_df["RV_next"]

har_model = sm.OLS(y, X).fit()

beta_hat = har_model.params
beta_hat

const    0.000614
RV_d     0.375803
RV_w     0.316711
RV_m     0.177906
dtype: float64

## 4. OLS inference

Compute residual variance, coefficient covariance, standard errors, and t-statistics explicitly.


In [10]:
epsilon_hat = y - X @ beta_hat

har_df["epsilon_hat"] = epsilon_hat

epsilon_hat

0      -0.000759
1       0.001116
2       0.000897
3      -0.001207
4      -0.000615
          ...   
1143    0.000190
1144   -0.000211
1145   -0.001005
1146   -0.000044
1147   -0.003265
Length: 1148, dtype: float64

In [11]:
n = len(y)
p = X.shape[1]

sigma_hat_squared = (epsilon_hat.T @ epsilon_hat) / (n - p)

sigma_hat_squared

np.float64(1.2600297932002032e-06)

In [12]:
cov_beta_hat = sigma_hat_squared * np.linalg.inv(X.T @ X)

cov_beta_hat = pd.DataFrame(
    cov_beta_hat,
    index=X.columns,
    columns=X.columns
)

cov_beta_hat

,const,RV_d,RV_w,RV_m
const,2.735906e-08,-3.320537e-07,-1.312891e-07,-0.000005
RV_d,-3.320537e-07,1.173581e-03,-1.166704e-03,0.000063
RV_w,-1.312891e-07,-1.166704e-03,2.947271e-03,-0.001753
RV_m,-5.046746e-06,6.301782e-05,-1.753420e-03,0.002749


In [13]:
se_beta_hat = np.sqrt(np.diag(cov_beta_hat)).reshape(-1, 1)

se_beta_hat = pd.DataFrame(
    se_beta_hat,
    index=X.columns,
    columns=["SE"]
)

se_beta_hat

,SE
const,0.000165
RV_d,0.034258
RV_w,0.054289
RV_m,0.052435


In [14]:
t_beta_hat = beta_hat.to_numpy().reshape(-1, 1) / se_beta_hat.to_numpy()

t_beta_hat = pd.DataFrame(
    t_beta_hat,
    index=X.columns,
    columns=["t_stat"]
)

# Final coefficient summary
results = pd.DataFrame({
    "coefficient": beta_hat,
    "SE": se_beta_hat["SE"],
    "t_stat": t_beta_hat["t_stat"]
})

results


,coefficient,SE,t_stat
const,0.000614,0.000165,3.714669
RV_d,0.375803,0.034258,10.969938
RV_w,0.316711,0.054289,5.833819
RV_m,0.177906,0.052435,3.392870


## 5. Newey--West (HAC) inference

Use HAC standard errors to allow for heteroskedasticity and serial correlation in the regression errors. The OLS coefficient estimates are unchanged.


In [15]:
nw_model = sm.OLS(y, X).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 5}
)

In [16]:
nw_model.params

const    0.000614
RV_d     0.375803
RV_w     0.316711
RV_m     0.177906
dtype: float64

In [17]:
nw_model.bse

const    0.000138
RV_d     0.049448
RV_w     0.078800
RV_m     0.079329
dtype: float64

In [18]:
nw_model.tvalues

const    4.447385
RV_d     7.599926
RV_w     4.019168
RV_m     2.242641
dtype: float64

In [19]:
nw_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                RV_next   R-squared:                       0.456
Model:                            OLS   Adj. R-squared:                  0.455
Method:                 Least Squares   F-statistic:                     307.8
Date:                Fri, 09 Oct 2026   Prob (F-statistic):          1.84e-146
Time:                        11:12:46   Log-Likelihood:                 6170.5
No. Observations:                1148   AIC:                        -1.233e+04
Df Residuals:                    1144   BIC:                        -1.231e+04
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.000      4.447      0.000       0.000       0.001
RV_d           0.3758      0.049      7.600      0.000       0.279       0.473
RV_w           0.3167      0.079      4.019      0.000       0.162       0.471
RV_m           0.1779      0.079      2.243      0.025       0.022       0.333
==============================================================================
Omnibus:                      426.803   Durbin-Watson:                   2.014
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2342.850
Skew:                           1.628   Prob(JB):                         0.00
Kurtosis:                       9.195   Cond. No.                     2.10e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 5 lags and without small sample correction
[2] The condition number is large, 2.1e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

## 6. Unrestricted AR(22)

Estimate 22 separate daily volatility lags as a less parsimonious benchmark for the HAR lag structure.


In [20]:
ar_data = pd.DataFrame()

for lag in range(22):
    ar_data[f"RV_lag_{lag}"] = rv_daily["RV"].shift(lag)

ar_data["RV_next"] = rv_daily["RV"].shift(-1)

ar_data = ar_data.dropna()

In [21]:
ar_data

,RV_lag_0,RV_lag_1,RV_lag_2,RV_lag_3,RV_lag_4,RV_lag_5,RV_lag_6,RV_lag_7,RV_lag_8,RV_lag_9,...,RV_lag_13,RV_lag_14,RV_lag_15,RV_lag_16,RV_lag_17,RV_lag_18,RV_lag_19,RV_lag_20,RV_lag_21,RV_next
21,0.003628,0.003922,0.003745,0.004547,0.003934,0.003578,0.003785,0.003556,0.003427,0.003611,...,0.004108,0.003821,0.003783,0.004341,0.003976,0.004028,0.004058,0.003982,0.004128,0.003159
22,0.003159,0.003628,0.003922,0.003745,0.004547,0.003934,0.003578,0.003785,0.003556,0.003427,...,0.003967,0.004108,0.003821,0.003783,0.004341,0.003976,0.004028,0.004058,0.003982,0.004801
23,0.004801,0.003159,0.003628,0.003922,0.003745,0.004547,0.003934,0.003578,0.003785,0.003556,...,0.003297,0.003967,0.004108,0.003821,0.003783,0.004341,0.003976,0.004028,0.004058,0.005221
24,0.005221,0.004801,0.003159,0.003628,0.003922,0.003745,0.004547,0.003934,0.003578,0.003785,...,0.003790,0.003297,0.003967,0.004108,0.003821,0.003783,0.004341,0.003976,0.004028,0.003379
25,0.003379,0.005221,0.004801,0.003159,0.003628,0.003922,0.003745,0.004547,0.003934,0.003578,...,0.003611,0.003790,0.003297,0.003967,0.004108,0.003821,0.003783,0.004341,0.003976,0.003239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1164,0.003825,0.003495,0.004032,0.003263,0.005132,0.005093,0.002933,0.003361,0.003706,0.004401,...,0.003932,0.003418,0.003169,0.003200,0.003767,0.003755,0.003820,0.003238,0.002857,0.004156
1165,0.004156,0.003825,0.003495,0.004032,0.003263,0.005132,0.005093,0.002933,0.003361,0.003706,...,0.004058,0.003932,0.003418,0.003169,0.003200,0.003767,0.003755,0.003820,0.003238,0.003828
1166,0.003828,0.004156,0.003825,0.003495,0.004032,0.003263,0.005132,0.005093,0.002933,0.003361,...,0.003494,0.004058,0.003932,0.003418,0.003169,0.003200,0.003767,0.003755,0.003820,0.002951
1167,0.002951,0.003828,0.004156,0.003825,0.003495,0.004032,0.003263,0.005132,0.005093,0.002933,...,0.004026,0.003494,0.004058,0.003932,0.003418,0.003169,0.003200,0.003767,0.003755,0.003507


In [22]:
X_ar = sm.add_constant(ar_data.drop(columns="RV_next"))
y_ar = ar_data["RV_next"]

ar22_model = sm.OLS(y_ar, X_ar).fit()

## 7. Compare HAR with AR(22)

Use the nested-model F-test and AIC/BIC to compare the parsimonious HAR specification with the unrestricted AR(22).


In [23]:
RSS_HAR = har_model.ssr
RSS_AR22 = ar22_model.ssr

n = ar22_model.nobs
q = 19
k_U = 23

F = ((RSS_HAR - RSS_AR22) / q) / (RSS_AR22 / (n - k_U))
F

np.float64(2.488870303594737)

In [24]:
p_value = 1 - f.cdf(F, q, n - k_U)

F, p_value

(np.float64(2.488870303594737), np.float64(0.0003981138983514265))

In [25]:
har_aic = nw_model.aic
har_bic = nw_model.bic

In [26]:
ar22_aic = ar22_model.aic
ar22_bic = ar22_model.bic

In [27]:
print("HAR AIC:", har_aic)
print("AR22 AIC:", ar22_aic)

print("HAR BIC:", har_bic)
print("AR22 BIC:", ar22_bic)

HAR AIC: -12332.986832915125
AR22 AIC: -12342.25552182893
HAR BIC: -12312.803726607606
AR22 BIC: -12226.202660560702


## 8. One-step-ahead in-sample forecasts

Construct fitted one-day-ahead HAR values and evaluate them using RMSE and MAE.


In [28]:
rv_df = (
    rv_daily[["trading_date", "RV"]]
    .copy()
    .sort_values("trading_date")
    .reset_index(drop=True)
)

rv_df

,trading_date,RV
0,2022-01-03,0.004128
1,2022-01-04,0.003982
2,2022-01-05,0.004058
3,2022-01-06,0.004028
4,2022-01-07,0.003976
...,...,...
1165,2026-06-25,0.004156
1166,2026-06-26,0.003828
1167,2026-06-29,0.002951
1168,2026-06-30,0.003507


In [29]:
forecast_df = pd.DataFrame({
    "trading_date": rv_features["trading_date"].iloc[1:].to_numpy(),
    "RV_forecast": (X @ beta_hat).to_numpy()
})

rv_df = rv_df.merge(
    forecast_df,
    on="trading_date",
    how="left"
)
rv_df = rv_df.dropna(subset=["RV_forecast"]).reset_index(drop=True)
rv_df

,trading_date,RV,RV_forecast
0,2022-02-02,0.003159,0.003918
1,2022-02-03,0.004801,0.003685
2,2022-02-04,0.005221,0.004325
3,2022-02-07,0.003379,0.004586
4,2022-02-08,0.003239,0.003853
...,...,...,...
1143,2026-06-25,0.004156,0.003966
1144,2026-06-26,0.003828,0.004039
1145,2026-06-29,0.002951,0.003956
1146,2026-06-30,0.003507,0.003551


In [30]:
# Forecast error: actual RV minus forecast RV
rv_df["forecast_error"] = rv_df["RV"] - rv_df["RV_forecast"]

# Evaluate only rows containing both actual and forecast values
evaluation_df = rv_df.dropna(
    subset=["RV", "RV_forecast"]
).copy()

rmse = np.sqrt(np.mean(evaluation_df["forecast_error"] ** 2))
mae = np.mean(np.abs(evaluation_df["forecast_error"]))

forecast_metrics = pd.DataFrame({
    "Metric": ["RMSE", "MAE"],
    "Value": [rmse, mae]
})

forecast_metrics

,Metric,Value
0,RMSE,0.001121
1,MAE,0.000789


## 9. Save reusable project data

The raw tick files are large and remain in `download/`. The compact derived datasets below are saved under `data/` so later notebooks can start from daily realized volatility rather than rebuilding it from tens of millions of ticks.


In [31]:
# Project folder is the parent of the existing raw-data download folder.
project_dir = download_dir.parent
data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 1) Daily realized-volatility series: main reusable input for later OOS work
rv_daily.to_csv(
    data_dir / "usdchf_daily_realized_volatility.csv",
    index=False
)

# 2) Regression-ready HAR dataset
har_df[
    ["trading_date", "RV_next", "RV_d", "RV_w", "RV_m"]
].to_csv(
    data_dir / "usdchf_har_dataset.csv",
    index=False
)

# 3) Current in-sample HAR forecasts and forecast errors
rv_df[
    ["trading_date", "RV", "RV_forecast", "forecast_error"]
].to_csv(
    data_dir / "usdchf_har_in_sample_forecasts.csv",
    index=False
)

print("Saved project data to:", data_dir)


Saved project data to: C:\Users\maria\Project\HAR-RV\data
